# Day 2: Data Objects in Databricks
## Scenario: Cyntexa Sales Domain Setup - Basic Task 1

This notebook creates the foundation for the `sales` domain in **Unity Catalog** by setting up the catalog, schema, raw managed table, filtered view, and performing exploratory queries on sample datasets.

In [0]:
%sql
-- Step 1: Create Catalog and Schema
CREATE CATALOG IF NOT EXISTS cyntexa_dev;

USE CATALOG cyntexa_dev;

CREATE SCHEMA IF NOT EXISTS sales;

USE SCHEMA sales;

In [0]:
%sql
-- Step 2: Create Managed Table and Insert Sample Data
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.orders_raw (
    order_id INT,
    customer_id INT,
    order_date DATE,
    total_amount DECIMAL(10,2),
    order_status STRING
);

INSERT INTO cyntexa_dev.sales.orders_raw VALUES
(101, 10001, '2026-08-01', 150.00, 'COMPLETED'),
(102, 10002, '2026-08-02', 89.50, 'PENDING'),
(103, 10001, '2026-08-03', 200.25, 'COMPLETED'),
(104, 10003, '2026-08-04', 45.00, 'CANCELLED'),
(105, 10004, '2026-08-05', 310.00, 'COMPLETED'),
(106, 10002, '2026-08-06', 12.99, 'PROCESSING'),
(107, 10005, '2026-08-07', 500.00, 'COMPLETED'),
(108, 10003, '2026-08-08', 75.80, 'CANCELLED'),
(109, 10006, '2026-08-09', 120.40, 'COMPLETED'),
(110, 10004, '2026-08-10', 95.00, 'PENDING');

In [0]:
%sql
-- Step 3: Create View for Completed Orders
CREATE VIEW IF NOT EXISTS cyntexa_dev.sales.orders_view AS 
SELECT 
order_id,
customer_id,
order_date,
total_amount
FROM 
cyntexa_dev.sales.orders_raw
WHERE 
order_status = 'COMPLETED';

In [0]:
%sql
-- Step 4: Exploratory Analysis on Samples (TPC-H Dataset)

-- Query 1: Top 5 customers by total account balance
SELECT 
    c_custkey, 
    c_name, 
    c_acctbal
FROM 
    samples.tpch.customer
ORDER BY c_acctbal DESC
LIMIT 5;

In [0]:
%sql
-- Query 2: Count of orders by order status
SELECT o_orderstatus , COUNT(*) AS total_orders
FROM 
samples.tpch.orders
GROUP BY
o_orderstatus;

In [0]:
%sql
-- Query 3: Average account balance per market segment
SELECT c_mktsegment , ROUND(AVG(c_acctbal),2) AS avg_balance
FROM samples.tpch.customer
GROUP BY 
c_mktsegment;

## Intermediate Tasks: Advanced Object Controls & Analytics

This section introduces reusable logic via SQL UDFs, compares Unity Catalog **Managed** vs. **External** table storage mechanics, and builds multi-object analytical views.

In [0]:
%sql 
-- Step 5: SQL UDF to Mask Sensitive Customer IDs
-- Create the UDF to mask all but the last character/digit (or append a masked prefix)
CREATE OR REPLACE FUNCTION cyntexa_dev.sales.mask_customer_id(cust_id INT)
RETURNS STRING
RETURN CONCAT('10**',RIGHT(CAST(cust_id AS STRING),1));

-- Test the UDF in a SELECT query against orders_raw
SELECT 
    order_id,
    customer_id,
    cyntexa_dev.sales.mask_customer_id(customer_id) AS masked_customer_id,
    total_amount,
    order_status
    FROM 
    cyntexa_dev.sales.orders_raw;

In [0]:
%sql
DESCRIBE EXTENDED cyntexa_dev.sales.orders_raw

## Why Location is Blank for Unity Catalog Managed Tables
This is actually by design in Unity Catalog! Here's what's happening:

Unity Catalog **Managed Tables** (first case):

**Location is blank/hidden** - Unity Catalog `abstracts` away the storage path
`Is_managed_location`: true - This tells you UC is fully managing the storage


**Why?** 
- UC doesn't want you to directly interact with or depend on the underlying storage path. It's saying "Don't worry about where the data lives, we handle it"


**Benefit** - Complete portability, lifecycle management, and you never need to think about storage paths


In [0]:
%sql
-- Checking External Locations
SHOW EXTERNAL LOCATIONS;

# Why External Tables Don't Work in Databricks Free Edition

## What We Tried to Do
Create an **External Table** pointing to cloud storage and compare it with a **Managed Table** using `DESCRIBE EXTENDED`.

## What Happened
We got errors. The external table could **not** be created.

---

## Quick Concepts

### Managed Table
- Databricks **owns** the table and the data.
- Data is stored inside Databricks' internal storage.
- If you drop the table, **data is deleted too**.
- **Type:** `MANAGED`
- **Location:** 

### External Table
- You **own** the data. Databricks only stores the schema (column names, types).
- Data stays in **your cloud storage** (AWS S3, Azure Data Lake, Google Cloud Storage).
- If you drop the table, **only the schema is deleted**. The data files stay safe in your cloud storage.
- **Type:** `EXTERNAL`
- **Location:** `s3://...` or `abfss://...` or `gs://...`

---

## Why It Fails in Free Edition

Databricks Free / Trial workspace uses **Unity Catalog** with **Serverless Compute**.

Unity Catalog has strict security rules:

| Requirement | Free Edition Status |
|-------------|-------------------|
| External Location (registered cloud storage path) | ❌ Not available |
| Admin rights to create External Location | ❌ Not available |
| `dbfs:/` paths for external tables | ❌ Blocked by Unity Catalog |
| Legacy `hive_metastore` catalog | ❌ Disabled |

**Result:** There is **no way** to provide a valid `LOCATION` for an external table.

---

## The Error We Saw


In [0]:
%sql
-- Step 7: (Data Analyst) Create Customers Table & Customer Spend View

-- 1. Create a dummy customers table to join with orders
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.customers(
    customer_id INT,
    customer_name STRING,
    email STRING
);

-- Populate customers table
INSERT INTO cyntexa_dev.sales.customers VALUES
(10001, 'Alice Smith', 'alice@example.com'),
(10002, 'Bob Jones', 'bob@example.com'),
(10003, 'Charlie Brown', 'charlie@example.com'),
(10004, 'Diana Prince', 'diana@example.com'),
(10005, 'Evan Wright', 'evan@example.com');

In [0]:
%sql
-- 2. Build view joining completed orders with customers to calculate total spend
CREATE OR REPLACE VIEW cyntexa_dev.sales.customer_spend_view AS
SELECT 
c.customer_id,
c.customer_name,
c.email,
COUNT(o.order_id) AS total_completed_order,
COALESCE(SUM(o.total_amount),0.00) AS total_spend
FROM 
cyntexa_dev.sales.customers c
LEFT JOIN 
cyntexa_dev.sales.orders_view o
ON c.customer_id = o.customer_id
GROUP BY 
c.customer_id, 
c.customer_name, 
c.email;

In [0]:
%sql
-- Query the spend view
SELECT * FROM cyntexa_dev.sales.customer_spend_view ORDER BY total_spend DESC;

# Advanced Tasks: Lakehouse Architecture & Analytics

This section covers production-grade catalog design, dynamic data-masking strategies, and advanced windowing analytics on standard benchmarking data.

# Cyntexa Three-Level Namespace Plan

## What is a "Namespace"? 

Think of a namespace like a **filing system** in an office.

> **Without Namespace:** All papers are dumped in one big room. Finding anything is a mess.
>
> **With Namespace:** Papers are organized in cabinets → folders → files. Easy to find.

In Databricks, a **Three-Level Namespace** means:
```
Cabinet (Catalog) → Folder (Schema) → File (Table)
```

---

## The Three Levels Explained

### Level 1: Catalog (The Big Container)

**What it is:** The top-most box. It holds everything.

**Analogy:** Like a **separate building** for each department or purpose.

| Catalog Name | Purpose |
|-------------|---------|
| `cyntexa_dev` | Developers experiment here. Break things. Test code. |
| `cyntexa_staging` | Pre-production testing. QA team checks here. |
| `cyntexa_prod` | Live production data. Real business runs here. |

**Think of it as:**
- `dev` = Practice ground (cricket net)
- `staging` = Dress rehearsal (trial match)
- `prod` = Real match (IPL final)

### Level 2: Schema (The Business Domain)

**What it is:** Inside each catalog, you divide data by **business area**.

**Analogy:** Like **rooms** inside a building — each room for a different team.

| Schema Name | Team/Domain | What Data They Own |
|------------|-------------|-------------------|
| `sales` | Sales Team | Orders, customers, invoices |
| `marketing` | Marketing Team | Campaigns, leads, ads data |
| `finance` | Finance Team | Budgets, transactions, reports |
| `hr` | HR Team | Employees, payroll, attendance |
| `operations` | Operations Team | Inventory, logistics, suppliers |

### Level 3: Table (The Actual Data)

**What it is:** The actual tables where rows and columns of data live.

**Analogy:** Like **files** inside a room — each file has specific information.

| Table Name | What It Stores |
|-----------|---------------|
| `orders` | All customer orders |
| `customers` | Customer profiles |
| `campaigns` | Marketing campaigns |
| `leads` | Potential customers |
| `employees` | Staff information |
| `payroll` | Salary data |

---

## Full Path Example

```sql
-- Format: catalog.schema.table

-- Development environment
SELECT * FROM cyntexa_dev.sales.orders;

-- Staging environment (testing before going live)
SELECT * FROM cyntexa_staging.sales.orders;

-- Production environment (real live data)
SELECT * FROM cyntexa_prod.sales.orders;
```

**Same table name, but completely separate data in each catalog.**

---

## The Complete Cyntexa Plan

### Catalogs (3 Total)

| Catalog | Who Uses It | Permissions | Data Quality |
|---------|------------|-------------|--------------|
| `cyntexa_dev` | Data Engineers, Analysts | Open, flexible | Experimental, can be messy |
| `cyntexa_staging` | QA Team, Reviewers | Controlled | Tested, but not final |
| `cyntexa_prod` | Business Users, Dashboards | Strict, read-only for most | Gold standard, 100% clean |

### Schemas Per Catalog (5 Total)

| Schema | Owner Team | Example Tables |
|--------|-----------|---------------|
| `sales` | Sales Team | `orders`, `customers`, `invoices`, `products` |
| `marketing` | Marketing Team | `campaigns`, `leads`, `ad_spend`, `conversions` |
| `finance` | Finance Team | `budgets`, `transactions`, `tax_reports`, `revenue` |
| `hr` | HR Team | `employees`, `payroll`, `attendance`, `hiring` |
| `operations` | Operations Team | `inventory`, `suppliers`, `shipments`, `warehouses` |

---

## Why This Structure? (Justification)

### 1. Environment Isolation

**Problem:** If dev and prod are mixed, a developer's bad query can delete live customer data.

**Solution:** Separate catalogs mean:
- `cyntexa_dev` mein kuch bhi todo, `cyntexa_prod` safe hai
- No accidental damage to real business data

### 2. Business Domain Organization

**Problem:** Sales team ke tables, HR ke tables, sab ek jagah pe hain. Confusion hoti hai.

**Solution:** Schemas by domain:
- Sales team sirf `sales` schema dekhti hai
- HR team sirf `hr` schema dekhti hai
- Clean ownership, no confusion

### 3. Easy Promotion Pipeline

**Problem:** Code dev se prod mein kaise le jayein? Pata nahi chalta.

**Solution:** Same structure in all 3 catalogs:
```
cyntexa_dev.sales.orders → cyntexa_staging.sales.orders → cyntexa_prod.sales.orders
```
- Table names same hain
- Sirf catalog name change hota hai
- Easy to move code from dev → staging → prod

### 4. Security & Governance

**Problem:** Sabko sab kuch dikhta hai. Data leak ka risk.

**Solution:**
- `cyntexa_prod` mein strict permissions
- `cyntexa_dev` mein engineers ko full access
- `cyntexa_prod` mein analysts ko sirf read access
- Schema level bhi control: HR data sirf HR team ko

### 5. Scalability

**Problem:** Company badi hoti hai, tables 100 ho jati hain. Manage nahi hota.

**Solution:**
- Naya business domain aaya? Naya schema banao.
- Naya team aayi? Unko apna schema do.
- Structure clean rehta hai chahe 10 tables ho ya 10,000.

---

## Real-World Example: How Data Flows

### Step 1: Developer creates table in DEV
```sql
-- Data Engineer is experimenting
CREATE TABLE cyntexa_dev.sales.orders_new_format (
    order_id INT,
    customer_id INT,
    order_date DATE,
    total_amount DECIMAL(10, 2)
)
USING DELTA;
```

### Step 2: QA tests in STAGING
```sql
-- QA team validates the new table
SELECT * FROM cyntexa_staging.sales.orders_new_format;
-- Run tests, check data quality
```

### Step 3: Business uses in PROD
```sql
-- Dashboard reads from production
SELECT SUM(total_amount) 
FROM cyntexa_prod.sales.orders_new_format
WHERE order_date >= '2024-01-01';
```

**Same table name. Same columns. But three completely separate copies.**

---

## Comparison: Good vs Bad Organization

### ❌ Bad Organization (Flat Structure)
```sql
-- All tables in one place, no structure
SELECT * FROM orders;           -- Whose orders? Dev or prod?
SELECT * FROM sales_orders;     -- Random naming
SELECT * FROM orders_v2;        -- Version confusion
SELECT * FROM final_orders;     -- "Final" never is final
```

**Problems:**
- No idea which environment
- Table names are messy
- Anyone can break anything
- No ownership

### ✅ Good Organization (Three-Level Namespace)
```sql
-- Crystal clear what, where, and whose
SELECT * FROM cyntexa_prod.sales.orders;
SELECT * FROM cyntexa_dev.marketing.campaigns;
SELECT * FROM cyntexa_staging.finance.budgets;
```

**Benefits:**
- Environment clear hai (dev/staging/prod)
- Owner team clear hai (sales/marketing/finance)
- Table purpose clear hai
- Clean, professional, scalable

---

## Non-Technical Stakeholder Summary

> **"We organize our data like a well-run office. Each building (catalog) is for a different purpose — one for practice, one for testing, one for real work. Inside each building, every team (schema) has their own room with their own files (tables). This keeps data safe, clean, and easy to find."**

---

## Quick Reference Table

| Term | Level | Analogy | Example |
|------|-------|---------|---------|
| **Catalog** | 1 | Building / Stadium | `cyntexa_prod` |
| **Schema** | 2 | Room / Department | `sales`, `marketing` |
| **Table** | 3 | File / Document | `orders`, `customers` |
| **Full Path** | All 3 | Building.Room.File | `cyntexa_prod.sales.orders` |

---

## Bottom Line

> **Three-Level Namespace = Organized, Safe, Scalable Data**
>
> **Catalog** = Environment (dev/staging/prod)
> **Schema** = Business Team (sales/marketing/finance/hr)
> **Table** = Actual Data (orders/customers/campaigns)
>
> **Full Path:** `catalog.schema.table`
>
> **Example:** `cyntexa_prod.sales.orders`


# Data Masking Strategy

## What is Data Masking?

Data masking hides sensitive information so that people who should not see it, cannot see it.

> **Example:** A customer service agent sees a phone number as `+1-XXX-XXX-1234`. Only a manager sees the full number.

---

## Which Columns Need Masking?

Not every column needs hiding. Only columns with **personal or sensitive data** need masking.

| Column Type | Example Column | Why Mask It? | Masked Example |
|-------------|---------------|--------------|----------------|
| **Personal Names** | `full_name` | Identity protection | `J*** D***` |
| **Email Address** | `email` | Privacy & spam risk | `j***@email.com` |
| **Phone Number** | `phone` | Direct contact risk | `+1-XXX-XXX-1234` |
| **Credit Card** | `credit_card` | Fraud prevention | `****-****-****-1234` |
| **Bank Account** | `bank_account` | Financial theft risk | `**********4567` |
| **Social Security / ID** | `ssn`, `national_id` | Identity theft | `***-**-1234` |
| **Salary** | `salary` | Employee privacy | `Confidential` |
| **Home Address** | `address` | Physical safety | `123 XXXX Street` |
| **Date of Birth** | `dob` | Age discrimination | `198X-XX-XX` |
| **Medical Records** | `diagnosis` | Health privacy | `Restricted` |
| **Passwords** | `password_hash` | Security breach | `Never visible` |

**Rule of thumb:** If losing this data in a leak would hurt a person or the company, mask it.

---

## Role Tiers: Who Sees What?

Different people need different levels of access. We create **role tiers**.

| Tier | Role Name | What They See | Examples |
|------|-----------|---------------|----------|
| **Tier 0** | Admin / Data Owner | **Full unmasked data** | Full SSN, full salary, full emails |
| **Tier 1** | Manager / Team Lead | **Mostly unmasked** | Full customer names, masked credit cards |
| **Tier 2** | Analyst / Engineer | **Partially masked** | Masked phone, full purchase history |
| **Tier 3** | Support Agent | **Heavily masked** | Masked name, masked email, last 4 digits only |
| **Tier 4** | Public / External | **Fully masked or denied** | `Access Denied` or `XXXX-XXXX` |

---

## Simple Example: A Customer Table

Imagine a table called `customers`:

| customer_id | full_name | email | phone | credit_card | salary |
|-------------|-----------|-------|-------|-------------|--------|
| 101 | John Doe | john@email.com | 555-1234 | 4111-2222-3333-4444 | 80000 |

**What each tier sees:**

| Tier | full_name | email | phone | credit_card | salary |
|------|-----------|-------|-------|-------------|--------|
| **Admin** | John Doe | john@email.com | 555-1234 | 4111-2222-3333-4444 | 80000 |
| **Manager** | John Doe | john@email.com | 555-1234 | ****-****-****-4444 | 80000 |
| **Analyst** | J*** D*** | j***@email.com | 555-XXXX | ****-****-****-4444 | Confidential |
| **Support** | J*** D*** | j***@email.com | XXX-XXXX | ****-****-****-**** | Restricted |

---

## How It Works in Databricks

Databricks uses **Unity Catalog** to control who sees what.

### Method 1: Dynamic Views

Create a view that shows different data based on the user's role.

```sql
-- Create a secure view that masks data
CREATE VIEW cyntexa_prod.sales.customers_masked AS
SELECT
    customer_id,
    CASE 
        WHEN is_account_group_member('admins') THEN full_name
        WHEN is_account_group_member('managers') THEN full_name
        ELSE concat(left(full_name, 1), '***')
    END AS full_name,
    CASE 
        WHEN is_account_group_member('admins') THEN email
        ELSE concat(left(email, 1), '***@', split_part(email, '@', 2))
    END AS email,
    CASE 
        WHEN is_account_group_member('admins') THEN phone
        ELSE 'XXX-XXXX'
    END AS phone
FROM cyntexa_prod.sales.customers;
```

### Method 2: Column Masking (Unity Catalog)

Apply a mask directly on the column. The mask stays even when someone queries the raw table.

```sql
-- Add a mask to the salary column
ALTER TABLE cyntexa_prod.sales.customers
ALTER COLUMN salary SET MASKING POLICY mask_salary;
```

---

## Quick Checklist

Use this checklist when building your masking plan:

- [ ] List all tables that contain personal data
- [ ] Mark which columns are sensitive
- [ ] Decide who needs full data (usually very few people)
- [ ] Create role groups: Admin, Manager, Analyst, Support
- [ ] Build views or policies for each role group
- [ ] Test by logging in as each role and checking what they see
- [ ] Review and update the plan every 6 months

---

## Bottom Line

> **Mask sensitive columns. Give full access only to people who really need it. Everyone else sees a safe, hidden version.**
>
> **Three things to remember:**
> 1. **What to mask:** Personal, financial, and health data
> 2. **Who sees full data:** Only admins and owners
> 3. **How to do it:** Use Databricks views or column policies


In [0]:
%sql
--Rather than creating separate views with masked columns (which leads to view sprawl), Unity Catalog supports Column Masks directly on base tables using is_account_group_member().  

--is_account_group_member() is a built-in Databricks function that checks if the user currently running the query belongs to a specific group at the account level (across your entire Databricks workspace/organization).
-- Why is it used?
-- It is mostly used for Row and Column Level Security to control who can see sensitive data.

-- Step 1: Create dynamic masking function
CREATE OR REPLACE FUNCTION cyntexa_dev.sales.email_mask(email STRING)
RETURNS STRING
RETURN IF(
    is_account_group_member('compliance_officers') OR is_account_group_member('data_engineers'),email,
    CONCAT(LEFT(email, 1), '***@', SPLIT(email, '@')[1])
);

-- Step 2: Bind mask directly to the table column in Unity Catalog
ALTER TABLE cyntexa_dev.sales.customers 
ALTER COLUMN email SET MASK cyntexa_prod.sales.email_mask;

In [0]:
%sql
-- Top 5 Customers by Revenue per Region using CTEs and Window Functions
WITH regional_customer_revenue AS (
    -- CTE 1: Calculate total revenue per customer and join across regional tables
    SELECT 
        r.r_name AS region_name,
        c.c_custkey AS customer_id,
        c.c_name AS customer_name,
        ROUND(SUM(l.l_extendedprice * (1 - l.l_discount)),2) AS total_revenue
    FROM samples.tpch.customer c
    JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
    JOIN samples.tpch.lineitem l ON o.o_orderkey = l.l_orderkey
    JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
    JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
    GROUP BY 
        r.r_name, 
        c.c_custkey, 
        c.c_name
),
ranked_regional_customers AS (
    -- CTE 2: Apply a window function to rank customers within each region by revenue
    SELECT 
        region_name,
        customer_id,
        customer_name,
        total_revenue,
        DENSE_RANK() OVER(
            PARTITION BY region_name 
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM regional_customer_revenue
)
-- Final Output: Filter top 5 customers per region
SELECT 
    region_name,
    revenue_rank,
    customer_id,
    customer_name,
    total_revenue
FROM 
    ranked_regional_customers
WHERE 
    revenue_rank <= 5
ORDER BY 
    region_name ASC, 
    revenue_rank ASC;